# Module 5: Model Evaluation & Feature Analysis

Deep dive into the final XGBoost model with combined features (23 total).

## Goals:
1. Rank features by their ability to distinguish ALS from Control
2. Compare feature discriminability in Discovery vs Validation
3. Identify most reliable biomarkers
4. Understand model predictions at feature level

**Model:** XGBoost with 23 combined features (17 fragmentomics + 6 methylation)

**Performance:** AUC 0.750, Accuracy 78.6%, Recall 100%


## Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
import pickle
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from src.config import PROCESSED_DIR, FINAL_FEATURES, FINAL_MODEL_PARAMS, RESULTS_DIR



## 1. Train / Validate XGBOOST

In [2]:
# Load data
all_features = pd.read_csv(PROCESSED_DIR / 'all_features.csv')

# Split discovery vs validation
discovery_df = all_features[all_features['batch'] == 'discovery'].copy()
validation_df = all_features[all_features['batch'] == 'validation'].copy()

# Define feature sets
frag_summary = [
    'frag_mean', 'frag_median', 'frag_std', 'frag_iqr', 'frag_cv',
    'frag_q25', 'frag_q50', 'frag_q75', 'frag_skewness', 'frag_kurtosis',
    'frag_pct_very_short', 'frag_pct_short', 'frag_pct_mononucleosomal',
    'frag_pct_dinucleosomal', 'frag_pct_long',
    'frag_ratio_short_long', 'frag_ratio_mono_di'
]
frag_summary = [f for f in frag_summary if f in all_features.columns]

meth_summary = [
    'meth_mean_cpg', 'meth_median_cpg', 'meth_std_cpg',
    'meth_pct_high', 'meth_pct_low', 'meth_pct_intermediate',
    'regional_meth_mean', 'regional_meth_median', 'regional_meth_std'
]
meth_summary = [f for f in meth_summary if f in all_features.columns]

# Combine features
feature_set = frag_summary + meth_summary
print(f"Using {len(feature_set)} features: {feature_set}")

# Prepare training data
X_train = discovery_df[feature_set].fillna(discovery_df[feature_set].median()).values
y_train = (discovery_df['disease_status'] == 'als').astype(int).values

# Prepare validation data
X_val = validation_df[feature_set].fillna(validation_df[feature_set].median()).values
y_val = (validation_df['disease_status'] == 'als').astype(int).values

# Train XGBoost
model = xgb.XGBClassifier(**FINAL_MODEL_PARAMS)
model.fit(X_train, y_train)

# Save trained model
model_file = RESULTS_DIR / 'trained_xgb_model.pkl'
with open(model_file, 'wb') as f:
    pickle.dump(model, f)
print(f"✓ Saved trained model: {model_file}")

# Predict on validation set
y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

Using 23 features: ['frag_mean', 'frag_median', 'frag_std', 'frag_iqr', 'frag_cv', 'frag_q25', 'frag_q50', 'frag_q75', 'frag_skewness', 'frag_kurtosis', 'frag_pct_very_short', 'frag_pct_short', 'frag_pct_mononucleosomal', 'frag_pct_dinucleosomal', 'frag_pct_long', 'frag_ratio_short_long', 'frag_ratio_mono_di', 'meth_mean_cpg', 'meth_pct_high', 'meth_pct_low', 'meth_pct_intermediate', 'regional_meth_mean', 'regional_meth_std']
✓ Saved trained model: /Users/maggiebrown/Desktop/PrimaMente/wgbs_classifier/results/trained_xgb_model.pkl


In [3]:
# Metrics
metrics = {
    'auc': roc_auc_score(y_val, y_pred_proba),
    'accuracy': accuracy_score(y_val, y_pred),
    'precision': precision_score(y_val, y_pred, zero_division=0),
    'recall': recall_score(y_val, y_pred, zero_division=0),
    'f1': f1_score(y_val, y_pred, zero_division=0)
}

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)

# Predictions dataframe
predictions = validation_df[['sample_id', 'disease_status', 'age']].copy()
predictions['true_label'] = y_val
predictions['pred_proba'] = y_pred_proba
predictions['pred_label'] = y_pred
predictions['correct'] = (y_val == y_pred)

# Feature importances
feature_importances = pd.DataFrame({
    'feature': feature_set,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Model trained on discovery set")
print(f"Validation Metrics: AUC={metrics['auc']:.3f}, Accuracy={metrics['accuracy']:.3f}, Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}, F1={metrics['f1']:.3f}")

Model trained on discovery set
Validation Metrics: AUC=0.750, Accuracy=0.786, Precision=0.727, Recall=1.000, F1=0.842


## Rank features

In [5]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(model)

# Compute SHAP values for validation set
shap_values = explainer.shap_values(X_val)

# Convert to DataFrame for easier handling
shap_df = pd.DataFrame(shap_values, columns=feature_set)
shap_mean_abs = pd.DataFrame({
    'feature': feature_set,
    'mean_abs_shap': np.abs(shap_df).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("✓ SHAP values computed and summarized")
shap_mean_abs.head(10)



ValueError: could not convert string to float: '[5E-1]'